# HDA Digital Twin collections census



**Goal**: establish, collection by collection, which Digital Twin
collections in the HDA production catalog actually expose zarr assets
through the desp_cache route, and cross the result against the datasets
described on the
[Earth Data Hub climate dt 2 page](https://earthdatahub.destine.eu/collections/climate-dt-2).

For each DT collection the census records: the HTTP status of a plain items
query, the number of items returned, whether any item carries a `.zarr`
asset, whether that asset href goes through `desp_cache`, and the first line
of the error body when the query fails (a 400 here usually means the
collection is order based and expects search constraints instead of a plain
listing).

In [2]:
from getpass import getpass

import destinelab as deauth
import pandas as pd
import requests

HDA_STAC_API = "https://hda.data.destination-earth.eu/stac/v2"

DESP_USERNAME = input("DESP username: ")
DESP_PASSWORD = getpass("DESP password: ")
auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
assert access_token is not None, "Failed to obtain access token"
headers = {"Authorization": f"Bearer {access_token}"}

In [3]:
# fetch all collections; the ".DT_" filter avoids the MDT_ false positive
resp = requests.get(f"{HDA_STAC_API}/collections", headers=headers)
resp.raise_for_status()
dt_ids = [c["id"] for c in resp.json()["collections"] if ".DT_" in c["id"]]
print(f"{len(dt_ids)} DT collections to probe\n")

rows = []
for cid in dt_ids:
    r = requests.get(f"{HDA_STAC_API}/collections/{cid}/items", headers=headers)
    row = {"collection": cid, "items_http": r.status_code, "n_items": None,
           "has_zarr": False, "via_desp_cache": False, "error": ""}
    if r.ok:
        feats = r.json().get("features", [])
        row["n_items"] = len(feats)
        for f in feats:
            for key, asset in f.get("assets", {}).items():
                if key.endswith(".zarr"):
                    row["has_zarr"] = True
                    if "desp_cache" in asset.get("href", ""):
                        row["via_desp_cache"] = True
    else:
        row["error"] = r.text[:120].replace("\n", " ")
    rows.append(row)
    print(f"{row['items_http']}  {cid}")

census = pd.DataFrame(rows)
census

40 DT collections to probe

400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_HIST_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_TPLUS2.0K_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_HIST_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_HIST_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLI

,collection,items_http,n_items,has_zarr,via_desp_cache,error
0,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1,400,NaN,False,False,"{""code"":""400"",""ticket"":""482d1590"",""description..."
1,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-N...,400,NaN,False,False,"{""code"":""400"",""ticket"":""82fd75e9"",""description..."
2,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_...,400,NaN,False,False,"{""code"":""400"",""ticket"":""6aaf9a4e"",""description..."
3,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_...,400,NaN,False,False,"{""code"":""400"",""ticket"":""fd1cf443"",""description..."
4,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""f1f5c169"",""description..."
5,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""4419ddf9"",""description..."
6,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""a6abc8d0"",""description..."
7,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CO...,400,NaN,False,False,"{""code"":""400"",""ticket"":""04e21e9f"",""description..."
8,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_HI...,400,NaN,False,False,"{""code"":""400"",""ticket"":""b7185b21"",""description..."
9,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_TP...,400,NaN,False,False,"{""code"":""400"",""ticket"":""eea125ef"",""description..."


In [4]:
# condensed view: which collections actually serve zarr through desp_cache
print("Collections WITH zarr via desp_cache:")
print(census[census.via_desp_cache]["collection"].to_string(index=False))
print()
print("Collections answering items but WITHOUT zarr assets:")
print(census[census.items_http.eq(200) & ~census.has_zarr]["collection"].to_string(index=False))
print()
print("Collections rejecting the plain items query:")
print(census[census.items_http.ne(200)][["collection", "items_http"]].to_string(index=False))

Collections WITH zarr via desp_cache:
    EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_ICON
EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_IFS-NEMO

Collections answering items but WITHOUT zarr assets:
Series([], )

Collections rejecting the plain items query:
                                                        collection  items_http
                  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1         400
              EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1         400
        EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1         400
         EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1         400
        EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1         400
   EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-FESOM.R1         400
    EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-NEMO.R1         400
     EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CONT_IFS-FESOM.R1         400
     EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_H